# 旧现金方向标签与v5标签同事件去重

只读两个封存Qlib Recorder；限2018–2020同一事件，比较符号和按实际名义beta增量归一的费后值排序。2021/2022不参与。本卡不拟合、不回测。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import json,hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/"research/stage1_v5_cash_label_route_identity_20261030"
plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
qlib.init(provider_uri=str(root/"data/qlib/stage1_event_guard_fullminute_20261028"),region="us",kernels=1)
loaded={};receipts={}
for role in ("old_source","v5_source"):
 spec=plan[role];source=root/spec["card"]
 assert sha(source/"summary.json")==spec["summary_sha256"] and sha(source/"verification_manifest.json")==spec["manifest_sha256"]
 if role=="v5_source":assert json.loads((source/"nas_binding.json").read_text())["evidence"]["sha256"]==spec["nas_archive_sha256"]
 o,receipt=load_verified_recorder_objects(source,expected_recorder_id=spec["recorder_id"],artifacts=["analysis/labels"],expected_manifest_sha256=spec["manifest_sha256"])
 loaded[role]=o["analysis/labels"];receipts[role]=receipt
old=loaded["old_source"]
new=loaded["v5_source"]
old=old.loc[old.trade_date.dt.year.isin(plan["discovery_years"])].copy()
new=new.loc[new.trade_date.dt.year.isin(plan["discovery_years"])].copy()
assert not old.trade_date.dt.year.isin(plan["reserved_years"]).any()
assert not new.trade_date.dt.year.isin(plan["reserved_years"]).any()
assert not old.duplicated(plan["join_key"]).any() and not new.duplicated(plan["join_key"]).any()
old_valid=old.loc[old.funded_change,plan["join_key"]+["local_advantage","cash_deployment_weight","decision_at","fill_at"]].copy()
new_valid=new.loc[new.soxx_funded,plan["join_key"]+["soxx_value","decision_at","fill_at"]].copy()
joined=old_valid.merge(new_valid,on=plan["join_key"],how="inner",validate="one_to_one",suffixes=("_old","_new"))
assert joined.decision_at_old.equals(joined.decision_at_new) and joined.fill_at_old.equals(joined.fill_at_new)
assert joined.cash_deployment_weight.gt(0).all()
joined["old_value_per_beta"]=joined.local_advantage/joined.cash_deployment_weight
joined["new_value_per_beta"]=joined.soxx_value/.10
joined["year"]=joined.trade_date.dt.year
annual=[]
for year in plan["discovery_years"]:
 a=old_valid.loc[old_valid.trade_date.dt.year.eq(year)];b=new_valid.loc[new_valid.trade_date.dt.year.eq(year)];part=joined.loc[joined.year.eq(year)]
 assert len(part)>0
 annual.append({"year":year,"old_funded_rows":len(a),"v5_soxx_funded_rows":len(b),"shared_funded_rows":len(part),
    "old_funded_days":int(a.trade_date.nunique()),"v5_soxx_funded_days":int(b.trade_date.nunique()),"shared_funded_days":int(part.trade_date.nunique()),
    "sign_agreement":float(np.mean(np.sign(part.local_advantage)==np.sign(part.soxx_value))),
    "normalized_spearman":float(part.old_value_per_beta.corr(part.new_value_per_beta,method="spearman")),
    "old_mean_beta_increment":float(part.cash_deployment_weight.mean()),
    "old_mean_local_value":float(part.local_advantage.mean()),"v5_mean_local_value":float(part.soxx_value.mean())})
annual=pd.DataFrame(annual)
annual.to_csv(card/"annual_identity.csv",index=False)
gate=bool(annual.loc[annual.year.isin([2019,2020]),"sign_agreement"].ge(.90).all() and
    annual.loc[annual.year.isin([2019,2020]),"normalized_spearman"].ge(.90).all())
audit={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),
    "decision":"not_admitted_same_cash_direction_label" if gate else "needs_deeper_label_attribution_no_model_admission",
    "source_readback_entries":{k:len(v) for k,v in receipts.items()},"discovery_years":plan["discovery_years"],
    "reserved_years_not_evaluated":plan["reserved_years"],"annual":annual.to_dict("records"),
    "necessary_same_direction_gate":gate,"new_model_fits":0,"new_native_accounts":0,"new_recorders":0,
    "limits":["Shared funded rows condition on future next-bar fill and do not define a live event selection rule.",
       "Old label resets at next open under original cap/cash floor; v5 is fixed-size cap-free before reset.",
       "2021/2022 outcomes were not compared; no SOXX account or alpha evidence."]}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"decision":audit["decision"],"annual":annual.to_dict("records")},ensure_ascii=False))

## 中文结论

只读2018–2020同事件旧SOXX现金买入与v5新标签：2019/2020共同成交41130/33654分钟、194/179日，费后价值同号96.77%/98.60%，按实际beta增量归一的Spearman0.9951/0.9963，预登记同义必要门通过，决定not_admitted_same_cash_direction_label。v5动作更大且不复位，但重叠状态下短期方向仍几乎等同旧失败现金路径，不重训旧15分钟收益或VWAP代理。2018部分年仅诊断；2021/2022目标值未用于选择。旧Recorder缺NAS上传收据、本地哈希读回；v5来源NAS冷读回通过。零拟合/账户，SOXX未过。下一机制寻找跨日现金与SOXL选择联合错误，不能只换短期价格方向窗口。